# YOLOv12n-seg Inference Demo

Run YOLOv12n-seg on images and visualize segmentation masks.


In [3]:
# Setup and Imports
import os
import sys
import importlib
from pathlib import Path

import torch
import numpy as np
import matplotlib.pyplot as plt

# Disable FlashAttention to avoid CUDA issues
os.environ['DISABLE_FLASH_ATTN'] = '1'

# Ultralytics imports
try:
    from ultralytics import YOLO
    import ultralytics.nn.modules.block as block_module
    block_module.USE_FLASH_ATTN = False
    importlib.reload(block_module)
    print("✓ Ultralytics imported, FlashAttention disabled")
except ImportError as e:
    print("ERROR: ultralytics module not found!")
    print("Please run: cd /home/tiehangz/proj/yolov12 && pip install -e .")
    raise

# Device info
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


FlashAttention is not available on this device. Using scaled_dot_product_attention instead.
✓ Ultralytics imported, FlashAttention disabled
Device: cuda
GPU: NVIDIA GeForce RTX 5070 Ti


In [4]:
# Load YOLOv12n-seg model
print("Loading YOLOv12n-seg model...")
model = YOLO('../model/yolov12n-seg.pt')  # Will download if not available
print("Model loaded successfully!")

# Find images to process
image_paths = []
# Check ultralytics assets
assets_dir = Path('ultralytics/assets')
if assets_dir.exists():
    for img_file in ['bus.jpg', 'zidane.jpg']:
        img_path = assets_dir / img_file
        if img_path.exists():
            image_paths.append(str(img_path))

# If no images found, use a test image URL
if not image_paths:
    print("No local images found, using test image from URL...")
    image_paths = ['https://ultralytics.com/images/bus.jpg']

print(f"\nProcessing {len(image_paths)} image(s)...")
print(f"Images: {image_paths}")

# Run inference
results = model.predict(image_paths, conf=0.25, imgsz=640)

print(f"\n✓ Inference complete! Processed {len(results)} image(s)")


Loading YOLOv12n-seg model...
Model loaded successfully!
No local images found, using test image from URL...

Processing 1 image(s)...
Images: ['https://ultralytics.com/images/bus.jpg']



AttributeError: 'JpegImageFile' object has no attribute '_im'

In [ ]:
# Visualize the first result with masks
if len(results) > 0:
    result = results[0]
    
    # Plot the result with masks
    annotated_img = result.plot(masks=True, boxes=True, labels=True, conf=True)
    
    # Display using matplotlib
    plt.figure(figsize=(12, 8))
    plt.imshow(annotated_img)
    plt.axis('off')
    plt.title(f'YOLOv12n-seg Segmentation Results\nImage: {Path(result.path).name}', fontsize=14)
    plt.tight_layout()
    
    # Save the visualization
    output_dir = Path('modification/outputs')
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / 'yolov12n_seg_demo.png'
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    print(f"\n✓ Visualization saved to: {output_path}")
    
    plt.show()
    
    # Print detection summary
    print(f"\nDetection Summary:")
    print(f"  Image: {result.path}")
    print(f"  Detections: {len(result.boxes) if result.boxes is not None else 0}")
    if result.masks is not None:
        print(f"  Masks: {len(result.masks)}")
        if result.boxes is not None:
            classes = result.boxes.cls.unique().cpu().tolist()
            class_names = [result.names[int(c)] for c in classes]
            print(f"  Classes detected: {class_names}")
else:
    print("No results to display")


In [ ]:
# Show individual mask detail if available
if len(results) > 0 and results[0].masks is not None and len(results[0].masks) > 0:
    result = results[0]
    
    print(f"Visualizing first mask...")
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Original image
    axes[0].imshow(result.orig_img)
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    # Mask overlay
    mask_overlay = result.plot(masks=True, boxes=False, labels=False)
    axes[1].imshow(mask_overlay)
    axes[1].set_title('Mask Overlay')
    axes[1].axis('off')
    
    # First mask only
    first_mask = result.masks.data[0].cpu().numpy()
    first_class = int(result.boxes.cls[0]) if result.boxes is not None else 0
    class_name = result.names[first_class]
    
    axes[2].imshow(first_mask, cmap='viridis')
    axes[2].set_title(f'First Mask\nClass: {class_name}')
    axes[2].axis('off')
    
    plt.tight_layout()
    
    # Save mask detail
    output_dir = Path('modification/outputs')
    output_dir.mkdir(parents=True, exist_ok=True)
    mask_output_path = output_dir / 'yolov12n_seg_mask_detail.png'
    plt.savefig(mask_output_path, dpi=150, bbox_inches='tight')
    print(f"✓ Mask detail saved to: {mask_output_path}")
    
    plt.show()
    
    # Print mask info
    print(f"\nMask Details:")
    print(f"  Mask shape: {first_mask.shape}")
    print(f"  Class: {class_name}")
    if result.boxes is not None:
        conf = float(result.boxes.conf[0])
        print(f"  Confidence: {conf:.3f}")
else:
    print("No masks found in results")
